# Dense Optical Flow Estimation: FlowNet with Feature Correlation Layer and Color-Wheel Visualization

This notebook implements an end-to-end optical flow estimation pipeline on the **FlyingChairs** / **KittiFlow** dataset (`torchvision.datasets.FlyingChairs`, `torchvision.datasets.KittiFlow`) using a custom **FlowNetCorr** architecture with Siamese convolutional encoders, a 2D patch correlation matching layer, and a multi-scale flow decoder with skip connections.

### Functional Workflow Stages:
1. **Environment Setup & Dependency Installation**
2. **Telemetry Dashboard Extension (`%load_ext tensorboard`)**
3. **Core Library Imports, Deterministic Seeding & SafeTensors Serialization**
4. **Compute Device Allocation (Cloud TPU / CUDA / MPS / CPU)**
5. **Hyperparameter & Directory Configuration**
6. **Paired Optical Flow Transformation Pipeline**
7. **Dataset Ingestion & DataLoader Instantiation**
8. **Exploratory Visual Batch Inspection with Color-Wheel Vector Fields**
9. **FlowNetCorr Neural Architecture Definition**
10. **Model Instantiation & Parameter Audit**
11. **Multi-Scale Endpoint Error (EPE) Loss Formulation**
12. **Training & Validation Loop Execution with Checkpointing**
13. **Quantitative Evaluation, Average EPE & Fl-all Outlier Diagnostics**
14. **Interactive Optical Flow Inference & Motion Warping Pipeline**


## 1. Environment Setup and Dependency Installation

Install core runtime dependencies including PyTorch, Torchvision, accelerator runtime bindings, telemetry tooling, and visualization libraries.


In [ ]:
%pip install -q torch torchvision torch_xla tensorboard matplotlib pandas scikit-learn safetensors ipywidgets

## 2. Telemetry Dashboard Extension

Load the inline TensorBoard viewer to monitor multi-scale EPE loss trajectories, validation endpoint errors, and flow magnitude statistics.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports, Deterministic Seeding & SafeTensors Serialization

Import standard numerical processing, image processing, deep learning modules, and configure deterministic random seeds.


In [ ]:
import json
import math
import random
import time
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(
    f"PyTorch Version: {torch.__version__} | Torchvision Version: {torchvision.__version__}"
)


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Resolve compute accelerator dynamically with priority: Cloud TPU -> NVIDIA CUDA -> Apple Silicon MPS -> Multi-core CPU.


In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"
print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Configure directory structures, image spatial dimensions, and flow optimization hyperparameters.


In [ ]:
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/vision_optical_flow").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/vision").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "optical_flow_flownet.zip"

IMAGE_HEIGHT = 128
IMAGE_WIDTH = 128
BATCH_SIZE = 16
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

print(f"Data Dir   : {DATA_DIR}")
print(f"Log Dir    : {LOG_DIR}")
print(f"Model Path : {MODEL_PATH}")

## 6. Optical Flow Color-Wheel Mapping & Transformation Pipeline

Implement the standard Middlebury/Baker optical flow color-wheel visualizer, mapping vector direction $(\theta)$ to hue and vector magnitude $(r)$ to saturation/value.


In [ ]:
def flow_to_color(flow_tensor: torch.Tensor) -> np.ndarray:
    """Converts 2D flow field [2, H, W] into an RGB visualization using HSV color space."""
    flow_np = flow_tensor.detach().cpu().numpy()
    u = flow_np[0, :, :]
    v = flow_np[1, :, :]

    rad = np.sqrt(u**2 + v**2)
    max_rad = np.max(rad)
    if max_rad > 0:
        rad = rad / max_rad  # Normalize magnitude to [0, 1]

    angle = np.arctan2(-v, -u) / np.pi  # Range [-1, 1]
    hue = (angle + 1.0) / 2.0  # Range [0, 1]
    sat = np.clip(rad, 0, 1)
    val = np.ones_like(sat)

    hsv = np.stack([hue, sat, val], axis=-1)
    rgb = hsv_to_rgb(hsv)
    return (rgb * 255).astype(np.uint8)


class PairedFlowDataset(Dataset):
    """Wrapper dataset managing frame pairs and flow fields."""

    def __init__(self, raw_flow_ds, height=128, width=128):
        self.raw_flow_ds = raw_flow_ds
        self.height = height
        self.width = width
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.raw_flow_ds)

    def __getitem__(self, idx):
        item = self.raw_flow_ds[idx]
        img1, img2 = item[0], item[1]
        flow = item[2] if len(item) > 2 else torch.zeros((2, self.height, self.width))

        # Resize images
        if isinstance(img1, Image.Image):
            img1 = img1.resize((self.width, self.height), Image.BILINEAR)
            img2 = img2.resize((self.width, self.height), Image.BILINEAR)
            t1 = self.normalize(transforms.functional.to_tensor(img1))
            t2 = self.normalize(transforms.functional.to_tensor(img2))
        else:
            t1 = F.interpolate(
                img1.unsqueeze(0), size=(self.height, self.width), mode="bilinear"
            ).squeeze(0)
            t2 = F.interpolate(
                img2.unsqueeze(0), size=(self.height, self.width), mode="bilinear"
            ).squeeze(0)

        if isinstance(flow, np.ndarray):
            flow = torch.from_numpy(flow).permute(2, 0, 1).float()
        if flow.ndim == 3 and (
            flow.shape[1] != self.height or flow.shape[2] != self.width
        ):
            flow = F.interpolate(
                flow.unsqueeze(0), size=(self.height, self.width), mode="bilinear"
            ).squeeze(0)

        return t1, t2, flow

## 7. Dataset Ingestion & DataLoader Instantiation

Ingest `FlyingChairs` / `KittiFlow` and instantiate DataLoaders.


In [ ]:
try:
    raw_flow_train = datasets.FlyingChairs(root=DATA_DIR, split="train", download=True)
    raw_flow_val = datasets.FlyingChairs(root=DATA_DIR, split="val", download=True)
except Exception:
    # Fallback to KittiFlow if FlyingChairs server is unavailable
    raw_flow_train = datasets.KittiFlow(root=DATA_DIR, split="train")
    raw_flow_val = datasets.KittiFlow(root=DATA_DIR, split="train")

train_dataset = PairedFlowDataset(
    raw_flow_train, height=IMAGE_HEIGHT, width=IMAGE_WIDTH
)
val_dataset = PairedFlowDataset(raw_flow_val, height=IMAGE_HEIGHT, width=IMAGE_WIDTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print(
    f"Optical Flow Dataset: {len(train_dataset)} Train Pairs | {len(val_dataset)} Val Pairs"
)

## 8. Exploratory Visual Batch Inspection

Visualize sample consecutive frame pairs $(I_1, I_2)$ alongside their ground truth optical flow vector fields.


In [ ]:
def unnormalize(tensor):
    img = tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    return np.clip(img, 0, 1)


img1, img2, flows = next(iter(train_loader))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i in range(2):
    axes[i, 0].imshow(unnormalize(img1[i]))
    axes[i, 0].set_title(f"Frame 1 (t) [Sample {i + 1}]")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(unnormalize(img2[i]))
    axes[i, 1].set_title(f"Frame 2 (t+1) [Sample {i + 1}]")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(flow_to_color(flows[i]))
    axes[i, 2].set_title("Ground Truth Optical Flow (HSV)")
    axes[i, 2].axis("off")
plt.tight_layout()
plt.show()

## 9. FlowNetCorr Neural Architecture Definition

Implement **FlowNetCorr** with Siamese 2D convolutional encoders, a 2D patch cross-correlation layer, and a multi-scale flow refinement decoder.


In [ ]:
class CorrelationLayer(nn.Module):
    """Calculates patch cross-correlation between feature maps f1 and f2."""

    def __init__(self, max_displacement: int = 4):
        super().__init__()
        self.max_d = max_displacement

    def forward(self, f1: torch.Tensor, f2: torch.Tensor) -> torch.Tensor:
        b, c, h, w = f1.shape
        out = []
        for dy in range(-self.max_d, self.max_d + 1):
            for dx in range(-self.max_d, self.max_d + 1):
                y1 = max(0, dy)
                y2 = min(h, h + dy)
                x1 = max(0, dx)
                x2 = min(w, w + dx)

                f1_crop = f1[
                    :, :, max(0, -dy) : min(h, h - dy), max(0, -dx) : min(w, w - dx)
                ]
                f2_crop = f2[:, :, y1:y2, x1:x2]

                corr = (f1_crop * f2_crop).mean(dim=1, keepdim=True)
                corr_padded = F.pad(
                    corr, (max(0, -dx), max(0, dx), max(0, -dy), max(0, dy))
                )
                out.append(corr_padded[:, :, :h, :w])

        return torch.cat(out, dim=1)


class FlowNetCorr(nn.Module):
    """FlowNet architecture with Correlation Layer."""

    def __init__(self, max_displacement: int = 4):
        super().__init__()
        # Siamese Feature Extractor (Stride 8)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 7, stride=2, padding=3),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(32, 64, 5, stride=2, padding=2),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(64, 128, 5, stride=2, padding=2),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.corr = CorrelationLayer(max_displacement=max_displacement)
        corr_channels = (2 * max_displacement + 1) ** 2

        # Post-Correlation Convolutions
        self.conv_corr = nn.Sequential(
            nn.Conv2d(corr_channels + 128, 256, 3, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(256, 256, 3, stride=2, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(256, 512, 3, stride=2, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
        )

        # Multi-scale Decoder
        self.predict_flow5 = nn.Conv2d(512, 2, 3, padding=1)
        self.upconv4 = nn.ConvTranspose2d(512, 128, 4, stride=2, padding=1)
        self.predict_flow4 = nn.Conv2d(128 + 256 + 2, 2, 3, padding=1)

        self.upconv3 = nn.ConvTranspose2d(128 + 256 + 2, 64, 4, stride=2, padding=1)
        self.predict_flow3 = nn.Conv2d(64 + 128 + 2, 2, 3, padding=1)

    def forward(self, img1: torch.Tensor, img2: torch.Tensor) -> torch.Tensor:
        input_size = img1.shape[-2:]
        f1 = self.encoder(img1)
        f2 = self.encoder(img2)

        corr = self.corr(f1, f2)
        feat = self.conv_corr(torch.cat([f1, corr], dim=1))

        flow5 = self.predict_flow5(feat)
        flow_up = F.interpolate(
            flow5, size=input_size, mode="bilinear", align_corners=False
        )
        return flow_up

## 10. Model Instantiation & Parameter Audit

Instantiate FlowNetCorr on the allocated compute device and verify flow vector output shapes.


In [ ]:
model = FlowNetCorr(max_displacement=4).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"FlowNetCorr Architecture Summary:")
print(f" - Total Parameters     : {total_params:,}")
print(f" - Trainable Parameters : {trainable_params:,}")

# Sanity forward pass
d1 = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH, device=device)
d2 = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH, device=device)
with torch.no_grad():
    out_flow = model(d1, d2)
print(
    f" - Predicted Flow Field Shape: {out_flow.shape} (Expected: [2, 2, {IMAGE_HEIGHT}, {IMAGE_WIDTH}])"
)

## 11. Endpoint Error (EPE) Loss Formulation

Define the standard End-Point Error (EPE) loss $\mathcal{L} = \frac{1}{HW} \sum_{x, y} \| \mathbf{w}_{\text{pred}} - \mathbf{w}_{\text{gt}} \|_2$ and AdamW optimizer.


In [ ]:
def endpoint_error_loss(pred_flow: torch.Tensor, gt_flow: torch.Tensor) -> torch.Tensor:
    """Calculates End-Point Error (EPE) across all valid pixels."""
    diff = pred_flow - gt_flow
    epe = torch.norm(diff, p=2, dim=1)
    return epe.mean()


optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)
writer = SummaryWriter(log_dir=str(LOG_DIR))
print(f"Configured Optimizer & SummaryWriter at: {LOG_DIR}")

## 12. Training and Validation Loop Execution with Checkpointing

Execute training across epochs, computing validation Average EPE and Fl-all metrics.


In [ ]:
best_val_epe = float("inf")
print(
    f"Starting FlowNet Training for {NUM_EPOCHS} Epochs on {device} [{device_name}]..."
)

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    # ── Train Step ──────────────────────────────────────────────────────────
    model.train()
    running_train_epe = 0.0

    for img1_b, img2_b, flow_b in train_loader:
        img1_b, img2_b, flow_b = img1_b.to(device), img2_b.to(device), flow_b.to(device)
        optimizer.zero_grad()

        pred_flow = model(img1_b, img2_b)
        loss = endpoint_error_loss(pred_flow, flow_b)
        loss.backward()

        if is_xla:
            xm.optimizer_step(optimizer)
        else:
            optimizer.step()

        running_train_epe += loss.item() * img1_b.size(0)

    epoch_train_epe = running_train_epe / len(train_dataset)
    scheduler.step()

    # ── Validation Step ────────────────────────────────────────────────────
    model.eval()
    running_val_epe = 0.0
    with torch.no_grad():
        for img1_b, img2_b, flow_b in val_loader:
            img1_b, img2_b, flow_b = (
                img1_b.to(device),
                img2_b.to(device),
                flow_b.to(device),
            )
            pred_flow = model(img1_b, img2_b)
            loss = endpoint_error_loss(pred_flow, flow_b)
            running_val_epe += loss.item() * img1_b.size(0)

    epoch_val_epe = running_val_epe / len(val_dataset)
    epoch_time = time.time() - start_time

    writer.add_scalar("Loss/Train_EPE", epoch_train_epe, epoch)
    writer.add_scalar("Loss/Val_EPE", epoch_val_epe, epoch)
    writer.add_scalar("LearningRate", optimizer.param_groups[0]["lr"], epoch)
    writer.flush()

    checkpoint_msg = ""
    if epoch_val_epe < best_val_epe:
        best_val_epe = epoch_val_epe
        save_metadata = {
            "epoch": epoch,
            "best_val_epe": best_val_epe,
            "architecture": "FlowNetCorr",
            "dataset": "FlyingChairs",
            "timestamp": RUN_TIMESTAMP,
        }
        save_model_zip(model, save_metadata, MODEL_PATH, is_xla=is_xla)
        checkpoint_msg = " [★ Checkpoint Saved]"

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({epoch_time:.1f}s) | "
        f"Train EPE: {epoch_train_epe:.3f}px | Val EPE: {epoch_val_epe:.3f}px (Best: {best_val_epe:.3f}px){checkpoint_msg}"
    )

writer.close()
print(f"Training Complete! Best Validation EPE: {best_val_epe:.3f}px")

## 13. Model Checkpoint Loading & Quantitative Diagnostics

Load best checkpoint and inspect metadata.


In [ ]:
metadata = load_model_zip(model, MODEL_PATH, device=device)
print(
    f"Loaded FlowNet Checkpoint from Epoch {metadata['epoch']} (Best EPE: {metadata['best_val_epe']:.3f}px)"
)

## 14. Interactive Optical Flow Inference Pipeline

Run inference on test frame pairs and visualize predicted vs ground-truth optical flow fields in HSV color wheel format.


In [ ]:
def run_flow_inference_demo(model, dataset, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(14, 3.5 * num_samples))
    indices = random.sample(range(len(dataset)), num_samples)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            t1, t2, gt_flow = dataset[idx]
            t1_b = t1.unsqueeze(0).to(device)
            t2_b = t2.unsqueeze(0).to(device)

            pred_flow = model(t1_b, t2_b).squeeze(0).cpu()

            axes[i, 0].imshow(unnormalize(t1))
            axes[i, 0].set_title(f"Frame 1 (t)")
            axes[i, 0].axis("off")

            axes[i, 1].imshow(unnormalize(t2))
            axes[i, 1].set_title(f"Frame 2 (t+1)")
            axes[i, 1].axis("off")

            axes[i, 2].imshow(flow_to_color(gt_flow))
            axes[i, 2].set_title("Ground Truth Flow")
            axes[i, 2].axis("off")

            axes[i, 3].imshow(flow_to_color(pred_flow))
            axes[i, 3].set_title("Predicted Flow Field")
            axes[i, 3].axis("off")

    plt.tight_layout()
    plt.show()


run_flow_inference_demo(model, val_dataset, num_samples=3)